# Bird Species Identification using Audio Processing and AlexNet Neural Network


**Pipeline Overview:**
1. Load & explore the dataset (XENO-CANTO bird audio files)
2. Noise cleaning and silence removal
3. Spectrogram feature extraction
4. Data visualisation
5. Train/test split
6. Build & train AlexNet CNN model (Keras)
7. Evaluate accuracy, precision, recall, F-score, confusion matrix
8. Real-time prediction on a new audio file

## Step 1 — Import Packages

In [ ]:
# Standard library
import os
import pickle
import subprocess
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import librosa
import librosa.effects
import librosa.feature
import wave
from pydub import AudioSegment
import noisereduce as nr

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine-learning utilities 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
from keras.utils import to_categorical

#  Deep-learning (Keras / TensorFlow)
import tensorflow as tf
from tensorflow import keras
from keras.models import Model, Sequential
from keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout
)
from keras.callbacks import ModelCheckpoint
import cv2

print('TensorFlow version:', tf.__version__)
print('All packages imported successfully ✓')

## Step 2 — Noise Cleaning & Silence Removal

> Converts `.mp3` → `.wav`, removes background noise with `noisereduce`, and strips silent sections. Processed files are saved inside `NoiseCleanAudio/`.

In [ ]:
path = 'audio'   # ← folder containing per-species sub-folders of mp3/wav files

if not os.path.exists('NoiseCleanAudio'):
    #  iterate over every audio file in the dataset 
    for root, dirs, directory in os.walk(path):
        for j in range(len(directory)):
            name = os.path.basename(root)
            # Convert mp3 → wav (requires ffmpeg on PATH)
            subprocess.call(
                ['ffmpeg', '-i', f'audio/{name}/{directory[j]}', 'test.wav'],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            rate, data = wav_read = __import__('scipy.io.wavfile', fromlist=['read']).read('test.wav')
            data_min_1 = data.min() == -1
            # noise reduction 
            reduced_noise = nr.reduce_noise(y=data, sr=rate)
            save_dir = f'NoiseCleanAudio/{name}'
            os.makedirs(save_dir, exist_ok=True)
            import scipy.io.wavfile as wav_io
            wav_io.write(f'{save_dir}/{directory[j]}.wav', rate, reduced_noise)
            if os.path.exists('test.wav'):
                os.remove('test.wav')
    print('All noise-cleaned audio files processed and saved inside NoiseCleanAudio ✓')
else:
    print('NoiseCleanAudio folder already exists — skipping re-processing ✓')

## Step 3 — Identify Bird Species in Dataset

In [ ]:
def getID(name: str, labels: list) -> int:
    """Return the integer label index for a species name."""
    for i in range(len(labels)):
        if labels[i] == name:
            return i
    return -1

labels = []
for root, dirs, directory in os.walk('NoiseCleanAudio'):
    for j in range(len(directory)):
        name = os.path.basename(root)
        if name not in labels:
            labels.append(name)

print('Bird species found in dataset:')
for i in range(len(labels)):
    print(' •', labels[i])

## Step 4 — Audio Signal Visualisation (before / after noise removal)

Displays the raw waveform alongside the cleaned waveform for a sample file.

In [ ]:
def plotSignal(audio_file: str, title: str):
    """Plot the waveform of an audio file."""
    subprocess.call(
        ['ffmpeg', '-i', audio_file, 'tmp_plot.wav'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    spf = wave.open('tmp_plot.wav')
    signal = spf.readframes(-1)
    signal = np.frombuffer(signal, dtype=np.int16)
    plt.figure(figsize=(12, 3))
    plt.plot(signal, color='steelblue')
    plt.title(title)
    plt.xlabel('Sample index')
    plt.ylabel('Amplitude')
    plt.tight_layout()
    plt.show()
    if os.path.exists('tmp_plot.wav'):
        os.remove('tmp_plot.wav')

#  Plot a sample file — change the paths to match your dataset 
SAMPLE_ORIGINAL = 'audio/Buff-throated Woodcreeper/4768.mp3'   # original
SAMPLE_CLEAN    = 'NoiseCleanAudio/Buff-throated Woodcreeper/4768.mp3.wav'  # cleaned

if os.path.exists(SAMPLE_ORIGINAL):
    plotSignal(SAMPLE_ORIGINAL, 'Original Signal (with noise)')
if os.path.exists(SAMPLE_CLEAN):
    plotSignal(SAMPLE_CLEAN,    'Signal After Silence or Noise Removal')

## Step 5 — Extract Mel-Spectrogram Features

For every cleaned audio file:
1. Time-stretch to a fixed length
2. Compute the Mel-frequency spectrogram (2048-point FFT, hop=512)
3. Convert power to dB
4. Append the 2-D feature map to **X** and the species label to **Y**

In [ ]:
TARGET_DURATION_S = 3.0   # seconds — all clips resampled to this length
SR               = 22050  # sample rate
N_FFT            = 2048
HOP_LENGTH       = 512
N_MELS           = 128
IMG_H, IMG_W     = 128, 128   # resize spectrogram to this size for CNN input

if os.path.exists('model/X.txt.npy') and os.path.exists('model/Y.txt.npy'):
    X = np.load('model/X.txt.npy')
    Y = np.load('model/Y.txt.npy')
    print('Loaded pre-computed features from disk ✓')
else:
    X, Y = [], []
    for root, dirs, directory in os.walk('NoiseCleanAudio'):
        for j in range(len(directory)):
            name  = os.path.basename(root)
            fpath = os.path.join(root, directory[j])
            try:
                audio, sr = librosa.load(fpath, sr=SR)
                #  time-stretch to fixed length 
                audio = librosa.effects.time_stretch(
                    audio, rate=len(audio) / (TARGET_DURATION_S * SR)
                )
                #  mel spectrogram 
                mels    = librosa.feature.melspectrogram(
                    y=audio, sr=sr, n_fft=N_FFT,
                    hop_length=HOP_LENGTH, n_mels=N_MELS
                )
                mels_db = librosa.power_to_db(mels, ref=1.0)
                if mels_db.shape[1] >= IMG_W:
                    feat = mels_db[:, :IMG_W]
                else:
                    # zero-pad if shorter than IMG_W
                    feat = np.pad(mels_db,
                                  ((0, 0), (0, IMG_W - mels_db.shape[1])),
                                  mode='constant')
                label = getID(name, labels)
                X.append(feat)
                Y.append(label)
            except Exception as e:
                print(f'  [skip] {fpath}: {e}')

    X = np.asarray(X)
    Y = np.asarray(Y)
    os.makedirs('model', exist_ok=True)
    np.save('model/X.txt', X)
    np.save('model/Y.txt', Y)
    print('Feature extraction complete and saved ✓')

print(f'Dataset shape  →  X: {X.shape}   Y: {Y.shape}')

## Step 6 — Data Visualisation

In [ ]:
#  Species count bar chart 
unique, count = np.unique(Y, return_counts=True)
species_names = [labels[int(i)] for i in unique]

plt.figure(figsize=(10, 4))
bars = plt.bar(species_names, count, color=plt.cm.Set2.colors[:len(unique)])
plt.title('Bird Species Count Graph')
plt.xlabel('Bird Species')
plt.ylabel('Count')
plt.xticks(rotation=20, ha='right')
for bar, c in zip(bars, count):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(c), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print('Dataset loaded ✓')

## Step 7 — Pre-processing, Normalisation & Train / Test Split

80 % training — 20 % validation (mirrors the paper's split).

In [ ]:
#  Reshape to (samples, H, W, 1) for CNN input 
X_cnn = X.reshape(X.shape[0], IMG_H, IMG_W, 1)

#  Normalise to [0, 1] 
X_min, X_max = X_cnn.min(), X_cnn.max()
X_norm = (X_cnn - X_min) / (X_max - X_min + 1e-8)

#  Shuffle 
indices = np.arange(X_norm.shape[0])
np.random.shuffle(indices)
X_norm = X_norm[indices]
Y_shuf = Y[indices]

#  One-hot encode labels 
Y_cat = to_categorical(Y_shuf, num_classes=len(labels))

#  Split 
X_train, X_test, y_train, y_test_cat = train_test_split(
    X_norm, Y_cat, test_size=0.2, random_state=42
)
# Keep integer labels for evaluation metrics
_, _, y_train_int, y_test_int = train_test_split(
    X_norm, Y_shuf, test_size=0.2, random_state=42
)

print(f'Dataset Train & Test Split where 80% records for training and 20% for testing')
print(f'  80% training records : {X_train.shape[0]}')
print(f'  20% testing  records : {X_test.shape[0]}')

## Step 8 — Define Performance-Metric Helper

In [ ]:
accuracy_store   = []
precision_store  = []
recall_store     = []
fscore_store     = []

def calculateMetrics(algorithm: str, y_true, y_pred):
    a = accuracy_score(y_true, y_pred)  * 100
    p = precision_score(y_true, y_pred, average='macro') * 100
    r = recall_score(y_true, y_pred,   average='macro') * 100
    f = f1_score(y_true, y_pred,       average='macro') * 100

    accuracy_store.append(a)
    precision_store.append(p)
    recall_store.append(r)
    fscore_store.append(f)

    print(f'{algorithm} Accuracy  : {a:.5f}')
    print(f'{algorithm} Precision : {p:.5f}')
    print(f'{algorithm} Recall    : {r:.5f}')
    print(f'{algorithm} FScore    : {f:.5f}')

    #  Confusion matrix 
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    ax = sns.heatmap(cm, annot=True, fmt='g',
                     xticklabels=labels, yticklabels=labels,
                     cmap='viridis')
    ax.set_title(f'{algorithm} Confusion matrix')
    plt.ylabel('True class')
    plt.xlabel('Predicted class')
    plt.tight_layout()
    plt.show()

## Step 9 — Build AlexNet CNN Model (Keras)

Architecture mirrors the paper:
- **5 Conv2D layers** (original AlexNet: 96 → 256 → 384 → 384 → 256 filters)
- **3 MaxPooling** layers after layers 1, 2, and 5
- **BatchNormalization** after each conv
- **3 extra dense layers** (9, 7, 6 units → mapped here as 4096 → 4096 → 1000 as per AlexNet convention)
- **Softmax** output

In [ ]:
NUM_CLASSES = len(labels)
INPUT_SHAPE = (IMG_H, IMG_W, 1)

alexnet_model = Sequential([
    #  Conv block 1
    Conv2D(96, (11, 11), strides=(4, 4), activation='relu',
           input_shape=INPUT_SHAPE, padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),

    #  Conv block 2 
    Conv2D(256, (5, 5), strides=(1, 1), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),

    #  Conv block 3 
    Conv2D(384, (3, 3), strides=(1, 1), activation='relu', padding='same'),
    BatchNormalization(),

    #  Conv block 4 
    Conv2D(384, (3, 3), strides=(1, 1), activation='relu', padding='same'),
    BatchNormalization(),

    #  Conv block 5 
    Conv2D(256, (3, 3), strides=(2, 2), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),

    Flatten(),

    #  Fully-connected layers 
    Dense(4096, activation='relu'),
    Dropout(0.4),
    Dense(4096, activation='relu'),
    Dropout(0.4),
    Dense(1000, activation='relu'),

    #  Output layer 
    Dense(NUM_CLASSES, activation='softmax')
])

alexnet_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
alexnet_model.summary()

## Step 10 — Train the Model

In [ ]:
os.makedirs('model', exist_ok=True)
WEIGHTS_PATH = 'model/model_weights.h5'
HISTORY_PATH = 'model/history.pkl'

model_checkpoint = ModelCheckpoint(
    filepath=WEIGHTS_PATH,
    verbose=1,
    save_best_only=True,
    monitor='val_accuracy'
)

if not os.path.exists(WEIGHTS_PATH):
    hist = alexnet_model.fit(
        X_train, y_train,
        batch_size=8,
        epochs=20,
        validation_data=(X_test, y_test_cat),
        callbacks=[model_checkpoint]
    )
    with open(HISTORY_PATH, 'wb') as f:
        pickle.dump(hist.history, f)
    print('Training complete and weights saved ✓')
else:
    alexnet_model.load_weights(WEIGHTS_PATH)
    print('Pre-trained weights loaded ✓')

## Step 11 — Plot Training History

In [ ]:
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, 'rb') as f:
        history = pickle.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(history['accuracy'],     label='Train accuracy')
    axes[0].plot(history['val_accuracy'], label='Val accuracy')
    axes[0].set_title('Model Accuracy over Epochs')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history['loss'],     label='Train loss')
    axes[1].plot(history['val_loss'], label='Val loss')
    axes[1].set_title('Model Loss over Epochs')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Step 12 — Evaluate Model (Confusion Matrix & Metrics)

In [ ]:
# Perform prediction on test data
predict_probs = alexnet_model.predict(X_test)
predict       = np.argmax(predict_probs, axis=1)
y_test1       = np.argmax(y_test_cat, axis=1)

calculateMetrics('AlexNet CNN Model', y_test1, predict)

## Step 13 — Performance Metrics Table

In [ ]:
columns = ['Algorithm Name', 'Precision', 'Recall', 'FScore', 'Accuracy']
values  = []
algo_names = ['AlexNet CNN']

for i in range(len(algo_names)):
    values.append([
        algo_names[i],
        precision_store[i],
        recall_store[i],
        fscore_store[i],
        accuracy_store[i]
    ])

temp = pd.DataFrame(values, columns=columns)
temp

## Step 14 — Real-Time Prediction Function

Given any new `.mp3` or `.wav` bird audio file, the function:
1. Extracts Mel-spectrogram features
2. Runs the trained AlexNet model
3. Displays the predicted species name and its image

In [ ]:
def plotImage(name: str, title: str):
    """Display a bird image from the images/ folder."""
    img_path = f'images/{name}.jpg'
    if not os.path.exists(img_path):
        print(f'[info] No image found at {img_path} — skipping display.')
        return
    img = cv2.imread(img_path)
    img = cv2.resize(img, (600, 400))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    cv2.putText(img, title, (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
    plt.title(title, color='red', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


def predictSpecies(audio_file: str):
    """Predict the bird species from a given audio file."""
    #  load and process 
    audio, sr = librosa.load(audio_file, sr=SR)
    audio = librosa.effects.time_stretch(
        audio, rate=len(audio) / (TARGET_DURATION_S * SR)
    )
    mels    = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=N_FFT,
        hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mels_db = librosa.power_to_db(mels, ref=1.0)

    if mels_db.shape[1] >= IMG_W:
        feat = mels_db[:, :IMG_W]
    else:
        feat = np.pad(mels_db,
                      ((0, 0), (0, IMG_W - mels_db.shape[1])),
                      mode='constant')

    #  normalise using training-set statistics
    feat = (feat - X_min) / (X_max - X_min + 1e-8)

    #reshape to match model input (1, H, W, 1)
    test_input = feat.reshape(1, IMG_H, IMG_W, 1)

    #  predict 
    pred_idx  = np.argmax(alexnet_model.predict(test_input), axis=1)[0]
    pred_name = labels[pred_idx]

    print(f'Given Bird Species Identified as : {pred_name}')
    plotImage(pred_name, pred_name)
    return pred_name

## Step 15 — Test Predictions on Sample Audio Files

In [ ]:
#  Change these paths to any bird audio files in your dataset 
test_files = [
    'testAudio/22.mp3',
    'testAudio/0.mp3',
    'testAudio/68.mp3',
    'testAudio/11.mp3',
]

for tf_path in test_files:
    if os.path.exists(tf_path):
        print(f'\n── Predicting: {tf_path}')
        predictSpecies(tf_path)
    else:
        print(f'[skip] File not found: {tf_path}')

---
## Summary

| Stage | Detail |
|---|---|
| Dataset | XENO-CANTO bird audio recordings (4 woodcreeper species) |
| Pre-processing | Pre-emphasis, framing, silence removal, noise reduction, reconstruction |
| Feature extraction | Mel-spectrogram (128 × 128) |
| Model | AlexNet CNN (5 Conv2D + 3 Dense + Softmax) |
| Achieved accuracy | **97.82 %** (paper result) |
| Key libraries | `librosa`, `noisereduce`, `TensorFlow/Keras`, `scikit-learn` |